# Hugging Face Transformers — inferencia directa

Hasta ahora hemos usado modelos a través de APIs (Anthropic, Ollama).
Con Transformers accedemos al modelo directamente en Python —
sin servidor intermedio, sin HTTP, el modelo corre en el mismo proceso.

Es más complejo pero te da control total:
ves exactamente qué entra, qué sale y cómo funciona por dentro.

## Qué veremos
- Cargar un modelo pequeño desde HF directamente en Python
- Entender qué es un tokenizer y cómo convierte texto en tokens
- Hacer inferencia y comparar con Ollama

In [2]:
from dotenv import load_dotenv
load_dotenv()
from transformers import AutoTokenizer

# Cargamos solo el tokenizer de un modelo pequeño
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

texto = "Hola, soy un desarrollador aprendiendo IA"

# Tokenizamos
tokens = tokenizer(texto)
print("IDs de tokens:", tokens["input_ids"])
print("Número de tokens:", len(tokens["input_ids"]))

# Decodificamos de vuelta a texto
decoded = tokenizer.decode(tokens["input_ids"])
print("Decodificado:", decoded)

# Ver cada token individualmente
for id in tokens["input_ids"]:
    print(f"  {id} → '{tokenizer.decode([id])}'")

IDs de tokens: [39, 5708, 11, 17797, 555, 748, 283, 2487, 7079, 2471, 10920, 72, 31110, 35229]
Número de tokens: 14
Decodificado: Hola, soy un desarrollador aprendiendo IA
  39 → 'H'
  5708 → 'ola'
  11 → ','
  17797 → ' soy'
  555 → ' un'
  748 → ' des'
  283 → 'ar'
  2487 → 'roll'
  7079 → 'ador'
  2471 → ' ap'
  10920 → 'rend'
  72 → 'i'
  31110 → 'endo'
  35229 → ' IA'


## Pipeline — inferencia de alto nivel

`pipeline` es la abstracción más simple de Transformers.
Envuelve tokenizer + modelo + decodificación en una sola llamada.

Es el equivalente a Ollama pero dentro de Python — sin servidor intermedio.
Más adelante bajaremos un nivel y usaremos `AutoModelForCausalLM` directamente
para tener control total sobre cada paso.

In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

resultado = generator(
    "La inteligencia artificial es",
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True
)

print(resultado[0]["generated_text"])

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 28251.98it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppres

La inteligencia artificial esperanza.
A.G.M.A.A.N.H.L.A is a research project of the National Science Foundation of the Netherlands.


## El idioma importa — distilgpt2 es un modelo inglés

distilgpt2 fue entrenado principalmente en inglés.
En español genera texto incoherente porque no conoce el idioma.
En inglés la calidad mejora notablemente — aunque sigue siendo un modelo pequeño.
Esto ilustra por qué el idioma de entrenamiento es crítico al elegir un modelo.

In [11]:
resultado_en = generator(
    "Artificial intelligence is",
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    pad_token_id=50256  # eos_token_id de GPT2
)

print(resultado_en[0]["generated_text"])

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial intelligence is a rapidly evolving technology, and it's worth checking out for yourself.




Image: Shutterstock

The next big thing to be excited about is that we've got a lot more of ways to go to work – from working on


## Modelo multilingüe — mismo código, mejor modelo

distilgpt2 solo funciona bien en inglés porque fue entrenado en inglés.
Cambiando el modelo en una sola línea tenemos acceso a uno multilingüe
que entiende español y sigue instrucciones.

Esto es la potencia de Transformers: la interfaz es siempre la misma

In [12]:
from transformers import pipeline

generator_es = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    pad_token_id=0
)

resultado = generator_es(
    "La inteligencia artificial es",
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
)

print(resultado[0]["generated_text"])

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 12922.14it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force clean

La inteligencia artificial es una tecnología que se está revolucionando en la industria, y a lo largo de este año se ha vuelto cada vez más popular. Aunque es importante tener en cuenta las ventajas de esta tecnología, también debemos recordar los ries


## AutoModelForCausalLM — control total

`pipeline` abstrae tokenizer + modelo + decodificación.
Aquí lo hacemos paso a paso para entender exactamente qué ocurre:

1. Tokenizamos el texto manualmente
2. Se lo pasamos al modelo
3. El modelo devuelve IDs de tokens
4. Decodificamos los IDs de vuelta a texto

Es más verbose pero ves exactamente qué entra y qué sale en cada paso.

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Paso 1: texto → tokens
texto = "La inteligencia artificial es"
inputs = tokenizer(texto, return_tensors="pt")
print("Tokens IDs:", inputs["input_ids"])

# Paso 2: el modelo genera nuevos tokens
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# Paso 3: tokens → texto
respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nRespuesta:", respuesta)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9108.63it/s]


Tokens IDs: tensor([[ 8747, 89645,  8792, 20443,  1531]])

Respuesta: La inteligencia artificial es un concepto que se refiere a la utilización de tecnologías de información y procesamiento de datos para realizar tareas de análisis, evaluación y predicción. Es una herramienta que puede ayudar en el manejo del inform
